In [ ]:
# =============================================
# IMPORTS
# =============================================
import os
import sys
import math

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel



if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"✓ Device: {device}")
print(f"✓ PyTorch: {torch.__version__}")

# Set the working directory to the project root
_nb_dir = os.path.abspath('')
if os.path.basename(_nb_dir) in ('experiments', 'examples', 'train'):
    os.chdir(os.path.dirname(_nb_dir))
sys.path.insert(0, os.path.join(os.getcwd(), 'PFNs'))
print(f"✓ Working directory: {os.getcwd()}")
# PFNs must be cloned (see setup.sh)
from pfns.train import train, MainConfig, OptimizerConfig, TransformerConfig, BatchShapeSamplerConfig
from pfns.model.encoders import EncoderConfig
from pfns.model.bar_distribution import BarDistributionConfig
from pfns.priors.prior import AdhocPriorConfig
from pfns.priors.fast_gp import get_batch as get_batch_for_gp

print("✓ PFNs loaded")

In [ ]:
# =============================================
# MODEL LOADING
# =============================================

# IMPORTANT: get_batch_for_gp_random_hps must be defined BEFORE torch.load
def get_batch_for_gp_random_hps(batch_size, seq_len, num_features, device="cpu", hyperparameters=None, **kwargs):
    random_hps = {
        "lengthscale": 0.3,
        "outputscale": 0.1,
        "noise":       1.0,
    }
    return get_batch_for_gp(batch_size, seq_len, num_features, device=device, hyperparameters=random_hps, **kwargs)


def load_for_inference(checkpoint_path, device='cpu'):
    """
    Loads an arbitrary PFN checkpoint for inference.
    Automatically detects nlayers from the checkpoint; works for 1L, 2L, 4L, 6L, ...
    """
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = checkpoint["config"]

    if "num_features" in config:
        # Old format
        num_features     = config["num_features"]
        max_dataset_size = config["max_dataset_size"]
        criterion        = checkpoint["criterion"]
        borders          = criterion.borders.tolist()
        nlayers          = config.get("nlayers", 6)
        get_batch_fn     = get_batch_for_gp
        prior_kwargs     = {"num_features": num_features, "hyperparameters": config.get("hps", {})}
    else:
        # New format (MainConfig)
        num_features     = config["priors"][0]["prior_kwargs"]["num_features"]
        max_dataset_size = config["batch_shape_sampler"]["max_seq_len"]
        borders          = config["model"]["criterion"]["borders"]
        nlayers          = config["model"].get("nlayers", 6)
        get_batch_fn     = get_batch_for_gp_random_hps
        prior_kwargs     = {"num_features": num_features, "hyperparameters": {}}
        criterion        = None

    model_config = MainConfig(
        priors=[AdhocPriorConfig(
            get_batch_methods=[get_batch_fn],
            prior_kwargs=prior_kwargs
        )],
        optimizer=OptimizerConfig("adamw", lr=0.0003),
        model=TransformerConfig(
            criterion=BarDistributionConfig(full_support=True, borders=borders),
            emsize=512, nhead=8, nhid=1024, nlayers=nlayers,
            features_per_group=1,
            attention_between_features=False,
            encoder=EncoderConfig(
                constant_normalization_mean=0.5,
                constant_normalization_std=math.sqrt(1/12)
            )
        ),
        batch_shape_sampler=BatchShapeSamplerConfig(
            batch_size=2, max_seq_len=max_dataset_size,
            min_num_features=num_features, max_num_features=num_features
        ),
        epochs=1, steps_per_epoch=1, num_workers=0,
    )

    dummy_result = train(model_config, device=device, reusable_config=False)
    model = dummy_result["model"]
    model.load_state_dict(checkpoint["model_state_dict"])
    if criterion is not None:
        model.criterion = criterion
    model.to(device)
    model.eval()

    epoch = checkpoint.get("epoch", "?")
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  ✓ {os.path.basename(checkpoint_path)}: nlayers={nlayers}, epoch={epoch}, {n_params:.2f}M params")
    return model, epoch


# =============================================
# MODEL CONFIGURATION - adjust paths as needed
# =============================================
MODEL_PATHS = {
    "1-layer": os.path.join("models", "pfn_rand_hps_1layer.pth"),
    "2-layer": os.path.join("models", "pfn_rand_hps_2layer.pth"),
    "4-layer": os.path.join("models", "pfn_rand_hps_4layer.pth"),
    "6-layer": os.path.join("models", "pfn_rand_hps_6layer.pth"),
    "8-layer": os.path.join("models", "pfn_rand_hps_8layer.pth"),
}

print("Loading models...")
MODELS = {}
for name, path in MODEL_PATHS.items():
    if os.path.exists(path):
        MODELS[name], _ = load_for_inference(path, device)
    else:
        print(f"  ⚠ Not found: {path}")

# Backward compatibility: loaded_model = first available model
loaded_model = next(iter(MODELS.values())) if MODELS else None

# =============================================
# GLOBAL SETTINGS
# =============================================
hps = {"noise": 1e-3, "outputscale": 1.0, "lengthscale": 0.3}

def reset_seed():
    import random
    s = random.randint(0, 2**32 - 1)
    torch.manual_seed(s)
    np.random.seed(s % (2**31))
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def get_batch_cpu(batch_size, seq_len, num_features, hyperparameters, device):
    batch = get_batch_for_gp(batch_size, seq_len, num_features,
                              device="cpu", hyperparameters=hyperparameters)
    from pfns.priors.prior import Batch
    return Batch(x=batch.x.to(device), y=batch.y.to(device), target_y=batch.target_y.to(device))

reset_seed()
print(f"\n✓ Loaded {len(MODELS)} models: {list(MODELS.keys())}")

## Experiment 3: Attention structure across layers

### Q1 - Label mixing vs. kernel computation (Step 3 from PFN-GP2.md)

The attention score in each layer combines both the coordinate $X_j$ and the label $Y_j$:

$$a_j^{(\ell)} \propto \exp\bigl(x_*^\top W_Q^{(\ell)} V_j\bigr), \quad V_j = (Y_j, X_j)^\top$$

A pure GP posterior depends only on $X_j$ through the RBF kernel. The PFN therefore does not implement kernel smoothing whenever $W_Q$ assigns a nonzero coefficient $\beta$ to $Y_j$.

**Hypothesis to test:** the network specializes across layers:
- Early layers: strong label influence, so $|\text{corr}(a^{(\ell)}, Y)|$ is large
- Late layers: stronger correlation with the RBF kernel, so $\text{corr}(a^{(\ell)}, k(x_*, X))$ grows

**For each layer $\ell$ and head $h$ we measure:**
$$\text{corr\_NW} = \text{Pearson}(a^{(\ell,h)}, \text{NW}(x_*))$$
$$\text{corr\_Y} = |\text{Pearson}(a^{(\ell,h)}, Y_{\text{train}})|$$

---

### Q3 - Minimum depth for hyperparameter identification (Step 4 from PFN-GP2.md)

Identifying the lengthscale $\ell$ from the context requires comparing distances between training points, which cannot be done within a single layer.

**Hypothesis to test:**
- 1-layer model: effective bandwidth is constant (matching the average $\ell$ of the prior)
- 2-layer model: bandwidth tracks the true $\ell$

**Effective bandwidth** from layer 0:
$$\text{bandwidth}(x_*) = \sum_j a_j^{(0)} \cdot |X_j - x_*|$$


In [ ]:
# =============================================
# UTILITY FUNCTIONS (attention hooks + NW weights)
# =============================================

def compute_nw_weights_q3(x_test_val, x_train_np, lengthscale):
    """Normalized RBF kernel weights for a single test point."""
    dist_sq = (x_train_np - x_test_val) ** 2
    w = np.exp(-dist_sq / (2 * lengthscale**2))
    return w / (w.sum() + 1e-10)


def extract_attention_weights_q3(model, train_x, train_y, test_x):
    """
    Captures the inputs to self_attn_between_items and computes attention weights manually.

    Inputs (without the batch dimension):
        train_x: [n_context, 1]
        train_y: [n_context]
        test_x:  [n_test, 1]

    Returns: list of length n_layers, each element [1, 1, n_heads, seq_len, n_context]
    Keys (K) are restricted to the training positions only, matching the model at inference.
    Attention weights in a test-point row therefore sum to 1 over the n_context positions.
    """
    n_context = train_x.shape[0]
    model.eval()
    layer_inputs = []

    def hook_fn(module, inputs, output):
        layer_inputs.append(inputs[0].detach().cpu())

    hooks = []
    attn_modules = []
    for name, module in model.named_modules():
        if 'self_attn_between_items' in name and 'self_attn_between_items.' not in name:
            hooks.append(module.register_forward_hook(hook_fn))
            attn_modules.append(module)

    with torch.no_grad():
        _ = model(train_x[None], train_y[None], test_x[None])

    for h in hooks:
        h.remove()

    all_attn = []
    for layer_idx, module in enumerate(attn_modules):
        if layer_idx >= len(layer_inputs):
            break
        x = layer_inputs[layer_idx]
        w_qkv = module.w_qkv.cpu()
        batch, features, seq_len, embed_dim = x.shape
        n_heads  = w_qkv.shape[1]
        head_dim = w_qkv.shape[2]
        x_flat = x.reshape(-1, embed_dim)
        W_q = w_qkv[0].permute(2, 0, 1)
        W_k = w_qkv[1].permute(2, 0, 1)
        Q = torch.matmul(x_flat, W_q.reshape(embed_dim, -1)).reshape(-1, n_heads, head_dim)
        K = torch.matmul(x_flat, W_k.reshape(embed_dim, -1)).reshape(-1, n_heads, head_dim)
        Q = Q.reshape(batch, features, seq_len, n_heads, head_dim).permute(0, 1, 3, 2, 4)
        K = K.reshape(batch, features, seq_len, n_heads, head_dim).permute(0, 1, 3, 2, 4)
        # Restrict keys to the training positions only (0..n_context-1).
        # Without this restriction the softmax would also include test tokens and
        # the weights in a test-point row would not sum to 1 over training positions.
        K_train = K[:, :, :, :n_context, :]
        scores = torch.matmul(Q, K_train.transpose(-2, -1)) / (head_dim ** 0.5)
        all_attn.append(F.softmax(scores, dim=-1))
    return all_attn

print("✓ Utility functions ready")

In [ ]:
# =============================================
# Q1: LABEL MIXING PER LAYER - functions
# =============================================

def run_label_mixing_per_layer(models_dict, n_instances, n_context, hps_q1, device, verbose=True):
    """
    For each model and each layer l computes:
      corr_nw[l] = mean correlation of attention weights with NW weights over heads and instances
      corr_y[l]  = mean absolute correlation of attention weights with labels Y

    Tested hypothesis: corr_nw increases with depth, corr_y decreases.
    """
    results = {}
    lengthscale = hps_q1['lengthscale']

    for model_name, model in models_dict.items():
        if verbose:
            print(f'  Model: {model_name}', flush=True)
        n_layers = model.transformer_layers.num_layers
        corr_nw_all = [[] for _ in range(n_layers)]
        corr_y_all  = [[] for _ in range(n_layers)]

        for i in range(n_instances):
            batch = get_batch_for_gp(
                batch_size=1, seq_len=n_context + 1, num_features=1,
                device='cpu', hyperparameters=hps_q1
            )
            train_x = batch.x[0, :n_context]
            train_y = batch.y[0, :n_context]
            test_x  = batch.x[0, n_context:n_context+1]

            try:
                attn_all = extract_attention_weights_q3(model, train_x, train_y, test_x)
            except Exception:
                continue

            tx_np = train_x.numpy().reshape(-1)
            ty_np = train_y.numpy().reshape(-1)
            test_val = float(test_x.numpy().reshape(-1)[0])
            nw_w = compute_nw_weights_q3(test_val, tx_np, lengthscale)

            for l, attn_l in enumerate(attn_all):
                if l >= n_layers:
                    break
                n_heads = attn_l.shape[2]
                for h in range(n_heads):
                    a = attn_l[0, 0, h, n_context, :n_context].detach().numpy()
                    if np.std(a) > 1e-8 and np.std(nw_w) > 1e-8:
                        c = float(np.corrcoef(a, nw_w)[0, 1])
                        if not np.isnan(c):
                            corr_nw_all[l].append(c)
                    if np.std(a) > 1e-8 and np.std(ty_np) > 1e-8:
                        c = abs(float(np.corrcoef(a, ty_np)[0, 1]))
                        if not np.isnan(c):
                            corr_y_all[l].append(c)

        results[model_name] = {
            'corr_nw':    np.array([np.mean(v) if v else np.nan for v in corr_nw_all]),
            'corr_nw_se': np.array([np.std(v)/np.sqrt(max(len(v),1)) for v in corr_nw_all]),
            'corr_y':     np.array([np.mean(v) if v else np.nan for v in corr_y_all]),
            'corr_y_se':  np.array([np.std(v)/np.sqrt(max(len(v),1)) for v in corr_y_all]),
            'n_layers':   n_layers,
        }
        if verbose:
            nw0 = results[model_name]['corr_nw'][0]
            y0  = results[model_name]['corr_y'][0]
            print(f'    L0: corr_nw={nw0:.3f}, corr_y={y0:.3f}')
    return results


def plot_label_mixing_per_layer(results):
    n_models = len(results)
    fig, axes = plt.subplots(1, n_models, figsize=(4*n_models, 5), sharey=True)
    if n_models == 1:
        axes = [axes]
    colors_nw = 'steelblue'
    colors_y  = 'tomato'

    for ax, (model_name, res) in zip(axes, results.items()):
        n = res['n_layers']
        x = np.arange(n)
        ax.plot(x, res['corr_nw'], 'o-', color=colors_nw, lw=2, ms=7,
                label='corr(attn, NW kernel)')
        ax.fill_between(x, res['corr_nw']-res['corr_nw_se'],
                           res['corr_nw']+res['corr_nw_se'], alpha=0.2, color=colors_nw)
        ax.plot(x, res['corr_y'], 's-', color=colors_y, lw=2, ms=7,
                label='|corr(attn, Y)|')
        ax.fill_between(x, res['corr_y']-res['corr_y_se'],
                           res['corr_y']+res['corr_y_se'], alpha=0.2, color=colors_y)
        ax.axhline(0, color='gray', ls=':', alpha=0.5)
        ax.set_xlabel('Layer l', fontsize=11)
        if ax is axes[0]:
            ax.set_ylabel('Mean correlation', fontsize=11)
        ax.set_title(model_name, fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels([f'L{i}' for i in x])
        ax.set_ylim(-0.2, 0.85)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle('Q1: Label mixing vs. kernel computation per layer', fontsize=13)
    plt.tight_layout()
    plt.show()

print("✓ Q1 functions ready")

In [ ]:
# =============================================
# RUN Q1
# =============================================

N_INSTANCES_Q1 = 200
N_CONTEXT_Q1   = 20
HPS_Q1 = {"noise": 1e-2, "outputscale": 1.0, "lengthscale": 0.3}

reset_seed()
print("Running Q1: label mixing per layer...")
q1_results = run_label_mixing_per_layer(MODELS, N_INSTANCES_Q1, N_CONTEXT_Q1, HPS_Q1, device)
plot_label_mixing_per_layer(q1_results)

## Q1 output

**Blue - corr(attn, NW kernel):** correlation of the attention weights with the Nadaraya-Watson weights (which depend only on the distance $x_* - X_j$).

**Red - |corr(attn, Y)|:** correlation of the attention weights with the label values $Y_j$.

Each panel shows one model; the x-axis is the layer index, the y-axis the mean correlation over heads and instances, with shaded standard-error bands.


In [ ]:
# =============================================
# Q3: EFFECTIVE BANDWIDTH VS. LENGTHSCALE - functions
# =============================================

def compute_effective_bandwidth(model, train_x, train_y, test_x, layer_idx=0):
    """
    Effective bandwidth from a given layer = mean distance |x_train - x_test|
    weighted by the attention weights (averaged over heads).

    bandwidth(x*) = mean_h  sum_j  attn_{j,h}^{(layer)} * |X_j - x*|
    """
    try:
        attn_all = extract_attention_weights_q3(model, train_x, train_y, test_x)
    except Exception:
        return np.nan
    if layer_idx >= len(attn_all):
        return np.nan

    attn_l    = attn_all[layer_idx]
    n_context = train_x.shape[0]
    n_heads   = attn_l.shape[2]
    tx_np = train_x.numpy().reshape(-1)
    te_np = test_x.numpy().reshape(-1)

    bw_list = []
    for ti in range(len(te_np)):
        dist = np.abs(tx_np - te_np[ti])
        head_bws = []
        for h in range(n_heads):
            a = attn_l[0, 0, h, n_context+ti, :n_context].detach().numpy()
            head_bws.append(float(np.dot(a, dist)))
        bw_list.append(np.mean(head_bws))
    return float(np.mean(bw_list))


def run_bandwidth_experiment(models_dict, lengthscales, n_instances, n_context, device, layer_idx=0):
    """
    For each model and each lengthscale: mean effective bandwidth from layer layer_idx.
    Returns: dict {model_name: {ell: (mean, se)}}
    """
    results = {name: {} for name in models_dict}
    for ell in lengthscales:
        hps_ell = {'noise': 1e-2, 'outputscale': 1.0, 'lengthscale': ell}
        print(f'  ell={ell:.2f}: ', end='', flush=True)
        for model_name, model in models_dict.items():
            bw_list = []
            for _ in range(n_instances):
                batch = get_batch_for_gp(
                    batch_size=1, seq_len=n_context+1, num_features=1,
                    device='cpu', hyperparameters=hps_ell
                )
                train_x = batch.x[0, :n_context]
                train_y = batch.y[0, :n_context]
                test_x  = batch.x[0, n_context:n_context+1]
                bw = compute_effective_bandwidth(model, train_x, train_y, test_x,
                                                  layer_idx=layer_idx)
                if not np.isnan(bw):
                    bw_list.append(bw)
            m = np.mean(bw_list) if bw_list else np.nan
            s = np.std(bw_list)/np.sqrt(max(len(bw_list),1)) if bw_list else np.nan
            results[model_name][ell] = (m, s)
            print(f'{model_name}={m:.3f}  ', end='', flush=True)
        print()
    return results


def plot_bandwidth_vs_lengthscale(results, lengthscales):
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.cm.tab10.colors
    for i, (model_name, bw_dict) in enumerate(results.items()):
        means = [bw_dict[ell][0] for ell in lengthscales]
        sems  = [bw_dict[ell][1] for ell in lengthscales]
        ax.errorbar(lengthscales, means, yerr=sems,
                    fmt='-o', color=colors[i], lw=2, ms=7, capsize=4, label=model_name)
    ax.plot(lengthscales, lengthscales, 'k--', alpha=0.4, lw=1.5, label='bandwidth = ℓ (ideal)')
    ax.set_xlabel('True lengthscale ℓ', fontsize=12)
    ax.set_ylabel('Effective bandwidth (layer 0)', fontsize=12)
    ax.set_title('Q3: Bandwidth adaptation to the true lengthscale', fontsize=13)
    ax.set_xscale('log')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()

print("✓ Q3 functions ready")

In [ ]:
# =============================================
# RUN Q3
# =============================================
# Compare only 1-layer vs. 2-layer: Q3 tests whether a single layer is sufficient

Q3_MODELS = {k: v for k, v in MODELS.items() if k in ('1-layer', '2-layer')}
if not Q3_MODELS:
    print("WARNING: Neither 1-layer nor 2-layer models found!")
else:
    LENGTHSCALES = [0.05, 0.1, 0.3, 0.8, 2.0]
    N_INSTANCES_Q3 = 100
    N_CONTEXT_Q3   = 20

    reset_seed()
    print('Running Q3: effective bandwidth vs. lengthscale...')
    q3_results = run_bandwidth_experiment(
        Q3_MODELS, LENGTHSCALES, N_INSTANCES_Q3, N_CONTEXT_Q3, device, layer_idx=0
    )
    plot_bandwidth_vs_lengthscale(q3_results, LENGTHSCALES)

    print('\n=== Q3 results: mean bandwidth ±SE ===')
    header = f'{"ℓ":>6}' + ''.join(f'  {n:>18}' for n in Q3_MODELS)
    print(header)
    print('-' * len(header))
    for ell in LENGTHSCALES:
        row = f'{ell:>6.2f}'
        for name in Q3_MODELS:
            m, s = q3_results[name][ell]
            row += f'  {m:>8.4f} ±{s:.4f}'
        print(row)

## Q3 output

**X axis (log):** true lengthscale $\ell$ used to generate the data.
**Y axis:** effective bandwidth from the **first layer** = $\sum_j a_j^{(0)} |X_j - x_*|$.

The black dashed line marks the ideal case bandwidth = ℓ.
